<a href="https://colab.research.google.com/github/kaljavi/HRM-LLM_Training/blob/main/HRM_LLM_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "transformers==4.41.2" "tokenizers==0.19.1" "huggingface_hub>=0.23.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting uninstall: transformers
    Found existing installation: transformers 4.54.1
    Uninstalling transformers-4.54.1:
      Successfully uninstalled transformers-4.54.1


In [ ]:
import os, math, json, random, io, datetime
from typing import Optional
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

from datasets import load_dataset
from transformers import T5Tokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

from huggingface_hub import HfApi, HfFolder, hf_hub_download
from google.colab import userdata

# ----------------------------
# Config
# ----------------------------
UPDATE_README = False
HF_REPO_ID = "SofiTesfay2010/HRM-LLM"
LOCAL_CHECKPOINT_PATH = "local_training_state.pt"
LOCAL_WEIGHTS_PATH = "pytorch_model.bin"
SEED = 42
MIXED_PRECISION = False  # start False for stability; turn True later
GRAD_ACCUM_STEPS = 1
NUM_EPOCHS = 5
BLOCK_SIZE = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_STEPS_RATIO = 0.1
MAX_HALT_STEPS = 8
PONDER_WEIGHT = 1e-2
MODEL_CONFIG = {"d_model": 512, "n_heads": 8, "d_ff": 2048, "dropout": 0.1}
T5_TOKENIZER_REPO = "t5-small"

# ----------------------------
# Utils
# ----------------------------
def set_seed(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ----------------------------
# Hub auth
# ----------------------------
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Hugging Face token found in Colab secrets.")
    HfFolder.save_token(HF_TOKEN)
    print("Login to Hugging Face Hub successful.")
except userdata.SecretNotFoundError:
    print("HF_TOKEN secret not found. Please create it in the Colab sidebar under 'Secrets'.")
    HF_TOKEN = None

# ----------------------------
# Tokenizer
# ----------------------------
print("Loading tokenizer...")
os.environ["TRANSFORMERS_NO_FAST_TOKENIZER"] = "1"
tokenizer = T5Tokenizer.from_pretrained(T5_TOKENIZER_REPO, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<pad>"})
tokenizer.padding_side = "left"
print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")

# ----------------------------
# Model layers
# ----------------------------
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    def forward(self, x):
        return self.weight * (x * torch.rsqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps))

class SwiGLUMuchPelu(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        activated = F.silu(self.w1(x)) * self.w2(x)
        return self.dropout(self.w3(activated))

class HRMBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d_model)
        self.mlp = SwiGLUMuchPelu(d_model, d_ff, dropout)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, attn_mask=None, key_padding_mask=None):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm,
                                attn_mask=attn_mask,
                                key_padding_mask=key_padding_mask,
                                need_weights=False)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.mlp(self.norm2(x)))
        return x

class HRMInner(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embeddings = nn.Embedding(config["vocab_size"], config["d_model"])
        self.dropout = nn.Dropout(config["dropout"])
        self.H_module = HRMBlock(config["d_model"], config["n_heads"], config["d_ff"], config["dropout"])
        self.L_module = HRMBlock(config["d_model"], config["n_heads"], config["d_ff"], config["dropout"])
    def forward(self, z_H, z_L, attn_mask=None, key_padding_mask=None):
        z_L_input = z_L + z_H
        z_L_new = self.L_module(z_L_input, attn_mask=attn_mask, key_padding_mask=key_padding_mask)
        z_H_input = z_H + z_L_new
        z_H_new = self.H_module(z_H_input, attn_mask=attn_mask, key_padding_mask=key_padding_mask)
        return z_H_new, z_L_new

class HierarchicalReasoningModel_ACTV1(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.inner_model = HRMInner(config)
        self.lm_head = nn.Linear(config["d_model"], config["vocab_size"], bias=False)
        self.halt_head = nn.Sequential(nn.Linear(config["d_model"], 1), nn.Sigmoid())
        self.max_steps = config["halt_max_steps"]
        self.ponder_loss_weight = config["ponder_loss_weight"]

    def forward(self, input_ids, labels=None, attention_mask=None):
        batch_size, seq_len = input_ids.shape
        device = input_ids.device
        z_L = self.inner_model.token_embeddings(input_ids)
        z_H = torch.zeros_like(z_L)

        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = (attention_mask == 0)

        mask_val = -1e4
        causal = torch.zeros((seq_len, seq_len), device=device)
        causal = causal.masked_fill(torch.triu(torch.ones_like(causal), diagonal=1).bool(), mask_val)

        halting_probs = torch.zeros((batch_size, seq_len, self.max_steps), device=device)
        remainders = torch.ones((batch_size, seq_len), device=device)
        total_z_H = 0.1 * z_L.clone()

        eps = 1e-6
        for step in range(self.max_steps):
            p_halt = self.halt_head(z_H).squeeze(-1).clamp(eps, 1 - eps)
            is_last = (step == self.max_steps - 1)
            halt_now_prob = torch.ones_like(p_halt) if is_last else p_halt
            contrib = (remainders * halt_now_prob).clamp(min=0.0, max=1.0)
            halting_probs[:, :, step] = contrib
            total_z_H = total_z_H + contrib.unsqueeze(-1) * z_H
            remainders = (remainders * (1 - p_halt)).clamp(min=eps, max=1.0)
            if torch.all(remainders < 1e-4):
                break
            z_H, z_L = self.inner_model(z_H, z_L, attn_mask=causal, key_padding_mask=key_padding_mask)

        logits = self.lm_head(total_z_H)
        loss = ponder_loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct = nn.CrossEntropyLoss()
            lm_loss = loss_fct(shift_logits.view(-1, self.config["vocab_size"]), shift_labels.view(-1))
            ponder_loss = torch.mean(torch.sum(halting_probs, dim=-1))
            loss = lm_loss + self.ponder_loss_weight * ponder_loss
        return {"loss": loss, "logits": logits, "ponder_loss": ponder_loss}

# ----------------------------
# Dataset
# ----------------------------
print("Loading dataset...")
raw_datasets = load_dataset("wikitext", "wikitext-2-raw-v1")
def tokenize_function(examples):
    texts = [t + (tokenizer.eos_token or "") for t in examples["text"]]
    return tokenizer(texts, add_special_tokens=False)

tokenized = raw_datasets.map(tokenize_function, batched=True, num_proc=2,
                             remove_columns=raw_datasets["train"].column_names)
train_dataset = tokenized["train"]
val_dataset = tokenized["validation"]

class LLMDataset(Dataset):
    def __init__(self, hf_dataset, block_size):
        all_token_ids = [tid for doc in hf_dataset["input_ids"] for tid in doc]
        self.examples = [all_token_ids[i:i+block_size]
                         for i in range(0, len(all_token_ids) - block_size + 1, block_size)]
    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return torch.tensor(self.examples[i], dtype=torch.long)

train_dataset = LLMDataset(train_dataset, BLOCK_SIZE)
val_dataset = LLMDataset(val_dataset, BLOCK_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# ----------------------------
# Model & optimizer
# ----------------------------
config = {
    "vocab_size": len(tokenizer),
    "d_model": MODEL_CONFIG["d_model"],
    "n_heads": MODEL_CONFIG["n_heads"],
    "d_ff": MODEL_CONFIG["d_ff"],
    "dropout": MODEL_CONFIG["dropout"],
    "halt_max_steps": MAX_HALT_STEPS,
    "ponder_loss_weight": PONDER_WEIGHT,
}

model = HierarchicalReasoningModel_ACTV1(config)

# ----------------------------
# FOR USING MULTIPLE GPUS:
# if torch.cuda.device_count() > 1:
#     print(f"Using {torch.cuda.device_count()} GPUs")
#     model = nn.DataParallel(model)
# ----------------------------

model.to(device)

with torch.no_grad():
    model.inner_model.token_embeddings.weight.normal_(0.0, 0.02)
    model.lm_head.weight.normal_(0.0, 0.02)

decay, no_decay = [], []
for n, p in model.named_parameters():
    if not p.requires_grad: continue
    if any(k in n.lower() for k in ["bias", "norm", "rmsnorm", "layernorm"]):
        no_decay.append(p)
    else:
        decay.append(p)
optimizer = AdamW([{"params": decay, "weight_decay": 0.01},
                   {"params": no_decay, "weight_decay": 0.0}], lr=LEARNING_RATE)

torch.save(model.state_dict(), LOCAL_WEIGHTS_PATH)           # SINGLE GPU
# torch.save(model.module.state_dict(), LOCAL_WEIGHTS_PATH)    # MULTI-GPU DataParallel
torch.save({
     "epoch": epoch,
     "optimizer_state_dict": optimizer.state_dict(),
     "scheduler_state_dict": scheduler.state_dict(),
     "global_step": global_step,
     "config": config
 }, LOCAL_CHECKPOINT_PATH)

    # Upload to Hub
    try:
        api = HfApi()
        print("Uploading weights to Hub...")
        api.upload_file(path_or_fileobj=LOCAL_WEIGHTS_PATH, path_in_repo=LOCAL_WEIGHTS_PATH,
                        repo_id=HF_REPO_ID, repo_type="model",
                        commit_message=f"Epoch {epoch} weights", token=HF_TOKEN)

        print("Uploading checkpoint to Hub...")
        api.upload_file(path_or_fileobj=LOCAL_CHECKPOINT_PATH, path_in_repo=LOCAL_CHECKPOINT_PATH,
                        repo_id=HF_REPO_ID, repo_type="model",
                        commit_message=f"Epoch {epoch} checkpoint", token=HF_TOKEN)

        print(f"Uploaded weights & checkpoint for epoch {epoch}.")
    except Exception as e:
        print(f"Upload failed: {e}")

print("Training complete.")


Using device: cuda
Hugging Face token found in Colab secrets.
Login to Hugging Face Hub successful.
Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


...

local_training_state.pt:   0%|          | 0.00/330M [00:00<?, ?B/s]

Resuming from epoch 35, global_step 49105


/tmp/ipython-input-1899896126.py:262: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(MIXED_PRECISION and device.type == "cuda"))


Training Epoch 35:   0%|          | 0/1403 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
